# 10. Corrida 5 x 5 en Colab (todos los modelos con el mismo código)

Este notebook repite las combinaciones con validación cruzada anidada de **5 folds externos y 5 internos** usando exactamente el código de `src/` del proyecto (`experiment_runner.run_experiments`), no una copia. Así todos los modelos, incluido SVM con `sklearn.svm.SVC`, se evalúan con el mismo solver y las mismas librerías.

**Cómo repartir el trabajo.** Cada sesión de Colab corre un subconjunto (`MODELS` y `TECHNIQUES`, celda 1) y guarda su propio CSV en Drive, así varias sesiones en paralelo no se pisan. Ejemplo de reparto (el costo medido de la corrida 3 x 3 era KNN 8.6 h, Random Forest 3.3 h, XGBoost 1.1 h, regresión logística 0.5 h, árbol 0.1 h y Naive Bayes 0.1 h, y con 5 x 5 se espera unas 4 veces más):

| Sesión | `MODELS` | `TECHNIQUES` |
|---|---|---|
| 1 | `naive_bayes`, `logistic_regression`, `decision_tree`, `xgboost` | todas |
| 2 a 5 | `knn` | una técnica por sesión |
| 6 a 9 | `random_forest` | una técnica por sesión |

Es seguro volver a ejecutar una sesión que se cortó: `resume=True` salta lo ya guardado.

**SVM.** Con CPU de Colab no es viable (en la corrida local de 3 x 3 una sola combinación tardó más de 4 horas). Para SVM se usa `07_svm_gpu_colab.ipynb` (cuML en GPU), ya ajustado a 5 x 5, y después se mide su diferencia con scikit-learn con `python -m src.svm_comparability`.

**Antes de empezar.** En Drive, carpeta `MyDrive/credit_card/`, subir `colab_package.zip` (contiene `src/`, `requirements.txt` y `data/processed/dataset_final.csv`).

In [ ]:
# Parámetros de ESTA sesión: cambiar solo esta celda.
MODELS = ["naive_bayes", "logistic_regression", "decision_tree", "xgboost"]
TECHNIQUES = None  # None = las 4 (none, smote, adasyn, class_weight); o p. ej. ["smote"]
METHODS = None     # None = los 4 (grid_search, random_search, bayesian_optuna, genetic_deap)

In [ ]:
import os, sys, zipfile
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/credit_card")
OUT_DIR = DRIVE_DIR / "resultados_5x5"
OUT_DIR.mkdir(parents=True, exist_ok=True)

WORK = Path("/content/credit_card")
if not (WORK / "src").exists():
    with zipfile.ZipFile(DRIVE_DIR / "colab_package.zip") as z:
        z.extractall(WORK)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("Paquete listo en", WORK, "| núcleos:", os.cpu_count())

In [ ]:
!pip install -q -r requirements.txt 2>&1 | tail -3
import sklearn, imblearn, xgboost, optuna, deap, numpy, pandas
print("scikit-learn", sklearn.__version__, "| imblearn", imblearn.__version__, "| xgboost", xgboost.__version__,
      "| optuna", optuna.__version__, "| deap", deap.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__)
print("Local (requirements.txt): scikit-learn 1.9.1, imbalanced-learn 0.14.2, xgboost 3.4.1, optuna 5.0.0, deap 1.4.3")

In [ ]:
import numpy as np
from src import config
from src.data_loader import load_processed_dataset, split_features_target
from src.preprocessing import remove_duplicate_rows, check_label_conflicts
from src.models import MODEL_REGISTRY
from src.experiment_runner import run_experiments, get_all_combinations

df, n_removed = remove_duplicate_rows(load_processed_dataset())
assert df.shape == (29965, 21), f"forma inesperada {df.shape}: ¿es el mismo dataset_final.csv?"
assert n_removed == 0
X, y = split_features_target(df)
np.random.seed(config.RANDOM_STATE)

assert (config.N_OUTER_FOLDS, config.N_INNER_FOLDS) == (5, 5), "src/config.py no está en 5 x 5"
# Paralelismo: todos los núcleos de esta máquina virtual, en el eje de evaluaciones de cada método.
config.OPTIMIZATION_N_JOBS = os.cpu_count()
print(f"X: {X.shape}, folds externos/internos: {config.N_OUTER_FOLDS}/{config.N_INNER_FOLDS}, n_jobs: {config.OPTIMIZATION_N_JOBS}")

In [ ]:
models = [MODEL_REGISTRY[m] for m in MODELS]
tag = "-".join(MODELS) + "__" + ("todas" if TECHNIQUES is None else "-".join(TECHNIQUES))
CHECKPOINT = OUT_DIR / f"experiments_5x5__{tag}.csv"

nj = config.OPTIMIZATION_N_JOBS
optimizer_kwargs_by_method = {
    "grid_search": {"n_jobs": nj},
    "random_search": {"n_iter": config.RANDOM_SEARCH_ITER, "n_jobs": nj},
    "bayesian_optuna": {"n_trials": config.OPTUNA_N_TRIALS, "n_jobs": nj},
    "genetic_deap": {"population_size": config.GA_POPULATION_SIZE, "n_generations": config.GA_N_GENERATIONS, "n_jobs": nj},
}
combos = get_all_combinations(models, TECHNIQUES, METHODS)
print(f"{len(combos)} combinaciones en esta sesión -> {CHECKPOINT}")

In [ ]:
table, results = run_experiments(
    X, y,
    models=models, techniques=TECHNIQUES, methods=METHODS,
    n_outer_folds=config.N_OUTER_FOLDS, n_inner_folds=config.N_INNER_FOLDS,
    optimizer_kwargs_by_method=optimizer_kwargs_by_method,
    checkpoint_path=CHECKPOINT, resume=True, n_jobs=1,
)
print(f"{len(results)} combinaciones nuevas en esta sesión; {len(table)} filas en {CHECKPOINT.name}")

## Resultado de esta sesión

In [ ]:
import pandas as pd
t = pd.read_csv(CHECKPOINT)
assert (t["n_outer_folds"] == 5).all() and (t["n_inner_folds"] == 5).all()
t[["model", "technique", "method", "f1_mean", "f1_std", "roc_auc_mean", "total_time_seconds"]].sort_values("f1_mean", ascending=False).round(4)

## Después de que terminen todas las sesiones

Descargar los CSV de `MyDrive/credit_card/resultados_5x5/` a `results/` del proyecto local y unirlos en `results/experiments_master_5x5.csv` (es la ruta que usa `src/config.py`):

```bash
python -m src.merge_session_results
```